# Retrieval filtering prototype

This notebook simulates the "mixed-quality organization text" setting:

- `AskUbuntuDupQuestions` acts as the in-distribution retrieval task.
- `FiQA2018` acts as the out-of-distribution retrieval task.
- `intfloat/e5-small-v2` is the embedding model.
- raw per-sample local intrinsic-dimension scores are used as the filter signal.

The notebook keeps retrieval evaluation self-contained with `pytrec_eval` because the installed `mteb` package currently has a `transformers` compatibility issue in this environment. The unsupervised diagnostics still reuse the repo code under `mteb_eval/src` and the local intrinsic-dimension helpers under `intrinsic_dim/`.

In [ ]:
from __future__ import annotations

import math
import random
import sys
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pytrec_eval
import seaborn as sns
import skdim
import torch
from datasets import load_dataset
from sentence_transformers import InputExample, SentenceTransformer
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss
from torch.utils.data import DataLoader


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "mteb_eval").exists() and (path / "intrinsic_dim").exists():
            return path
    raise FileNotFoundError("Could not find the repository root")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(REPO_ROOT / "mteb_eval") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "mteb_eval"))

from src.unsup_metrics import compute_metrics_retrieval, suffix_metrics
from intrinsic_dim.intrinsic_dim import (
    FAST_GLOBAL_ESTIMATORS,
    FAST_LOCAL_ESTIMATORS,
    compute_intrinsic_dim_global,
    compute_intrinsic_dim_local,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "intfloat/e5-small-v2"
DEVICE = "cpu"  # switch to "cuda" manually if your environment supports it

DATASET_SPECS = [
    {
        "task_name": "AskUbuntuDupQuestions",
        "dataset_id": "Samoed/AskUbuntuDupQuestions",
        "corpus_cap": 7500,
    },
    {
        "task_name": "FiQA2018",
        "dataset_id": "mteb/fiqa",
        "corpus_cap": 7500,
    },
]

ENCODE_BATCH_SIZE = 64
LOCAL_ID_NEIGHBORS = 50
LOCAL_ID_ESTIMATORS = tuple(name for name in FAST_LOCAL_ESTIMATORS if name in ("MLE", "MOM")) or ("MLE", "MOM")
KEEP_HIGH_SCORES = True
THRESHOLD_QUANTILES = np.linspace(0.05, 0.95, 19)
TOP_K = 100
EVAL_CUTOFFS = (1, 3, 5, 10, 100)
UNSUP_SAMPLE_FRACTION = 0.15
UNSUP_MIN_SAMPLE_SIZE = 256
UNSUP_SELECTED_METRICS = ["rankme", "coherence", "stable_rank", "intrinsic_dim_local_fast"]
DIAG_INTRINSIC_SAMPLE_SIZE = 1024

RUN_ROOT = REPO_ROOT / "results"
RUN_NAME = f"ood_local_metric_filtering_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR = RUN_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR

In [ ]:
import json
from dataclasses import dataclass
from typing import Any


@dataclass(frozen=True)
class RetrievalTaskConfig:
    task_name: str
    dataset_id: str
    corpus_cap: int


TASK_CONFIGS = [RetrievalTaskConfig(**spec) for spec in DATASET_SPECS]


def _dataset_configs(dataset_id: str) -> dict[str, tuple[str, str]]:
    if dataset_id == "Samoed/AskUbuntuDupQuestions":
        return {
            "queries": ("queries", "test"),
            "corpus": ("corpus", "test"),
            "qrels": ("qrels", "test"),
        }
    if dataset_id == "mteb/fiqa":
        return {
            "queries": ("queries", "queries"),
            "corpus": ("corpus", "corpus"),
            "qrels": ("default", "test"),
        }
    return {
        "queries": ("queries", "test"),
        "corpus": ("corpus", "test"),
        "qrels": ("default", "test"),
    }


def _first_existing(columns: Sequence[str], candidates: Sequence[str]) -> str:
    for col in candidates:
        if col in columns:
            return col
    raise KeyError(f"Could not find any of {candidates} in columns: {list(columns)}")


def _normalize_queries(ds: Any) -> pd.DataFrame:
    df = ds.to_pandas().copy()
    id_col = _first_existing(df.columns, ["id", "query-id", "query_id", "queryId"])
    text_col = _first_existing(df.columns, ["text", "query", "sentence", "content"])
    out = df[[id_col, text_col]].rename(columns={id_col: "query_id", text_col: "text"})
    out["query_id"] = out["query_id"].astype(str)
    out["text"] = out["text"].astype(str).str.strip()
    return out[out["text"] != ""].drop_duplicates("query_id").reset_index(drop=True)


def _normalize_corpus(ds: Any) -> pd.DataFrame:
    df = ds.to_pandas().copy()
    id_col = _first_existing(df.columns, ["id", "corpus-id", "corpus_id", "doc_id", "docid"])
    text_col = _first_existing(df.columns, ["text", "content"])
    columns = [id_col, text_col] + (["title"] if "title" in df.columns else [])
    out = df[columns].copy()

    def render_text(row: pd.Series) -> str:
        body = "" if pd.isna(row[text_col]) else str(row[text_col]).strip()
        if "title" in row.index and not pd.isna(row["title"]):
            title = str(row["title"]).strip()
            if title:
                return f"{title} {body}".strip()
        return body

    out["text"] = out.apply(render_text, axis=1)
    out = out.rename(columns={id_col: "doc_id"})[["doc_id", "text"]]
    out["doc_id"] = out["doc_id"].astype(str)
    out["text"] = out["text"].astype(str).str.strip()
    return out[out["text"] != ""].drop_duplicates("doc_id").reset_index(drop=True)


def _normalize_qrels(ds: Any) -> pd.DataFrame:
    df = ds.to_pandas().copy()
    q_col = _first_existing(df.columns, ["query-id", "query_id", "query"])
    d_col = _first_existing(df.columns, ["corpus-id", "corpus_id", "doc_id", "doc-id"])
    s_col = _first_existing(df.columns, ["score", "relevance"])
    out = df[[q_col, d_col, s_col]].rename(columns={q_col: "query_id", d_col: "doc_id", s_col: "score"})
    out["query_id"] = out["query_id"].astype(str)
    out["doc_id"] = out["doc_id"].astype(str)
    out["score"] = pd.to_numeric(out["score"], errors="coerce").fillna(0).astype(int)
    return out


def load_retrieval_tables(config: RetrievalTaskConfig) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    cfg = _dataset_configs(config.dataset_id)
    queries = _normalize_queries(load_dataset(config.dataset_id, cfg["queries"][0], split=cfg["queries"][1]))
    corpus = _normalize_corpus(load_dataset(config.dataset_id, cfg["corpus"][0], split=cfg["corpus"][1]))
    qrels = _normalize_qrels(load_dataset(config.dataset_id, cfg["qrels"][0], split=cfg["qrels"][1]))
    return queries, corpus, qrels


def build_qrels_map(qrels_df: pd.DataFrame) -> Dict[str, Dict[str, int]]:
    qrels_map: Dict[str, Dict[str, int]] = {}
    positives = qrels_df[qrels_df["score"] > 0]
    for qid, group in positives.groupby("query_id"):
        qrels_map[str(qid)] = {str(row.doc_id): int(row.score) for row in group.itertuples()}
    return qrels_map


def build_analysis_pool(
    task_name: str,
    queries_df: pd.DataFrame,
    corpus_df: pd.DataFrame,
    qrels_map: Dict[str, Dict[str, int]],
    corpus_cap: int,
    seed: int = SEED,
) -> pd.DataFrame:
    query_pool = queries_df.copy().reset_index(drop=True)
    query_pool = query_pool[query_pool["query_id"].isin(qrels_map.keys())].copy().reset_index(drop=True)
    query_pool["kind"] = "query"
    query_pool["item_id"] = query_pool["query_id"]
    query_pool["doc_id"] = pd.NA
    query_pool["prefix"] = "query: "
    query_pool["prefixed_text"] = query_pool["prefix"] + query_pool["text"].astype(str)

    required_doc_ids: set[str] = set()
    for qid in query_pool["query_id"]:
        required_doc_ids.update(qrels_map.get(qid, {}).keys())

    corpus_pool = corpus_df.copy().reset_index(drop=True)
    corpus_pool["is_required"] = corpus_pool["doc_id"].isin(required_doc_ids)
    required = corpus_pool[corpus_pool["is_required"]].copy().reset_index(drop=True)

    if len(required) >= corpus_cap:
        print(f"[{task_name}] required corpus docs ({len(required)}) exceed corpus_cap={corpus_cap}; keeping all required docs")
        corpus_pool = required.copy()
    else:
        extra_budget = corpus_cap - len(required)
        remaining = corpus_pool[~corpus_pool["is_required"]].copy().reset_index(drop=True)
        if extra_budget > 0 and len(remaining) > 0:
            extra = remaining.sample(n=min(extra_budget, len(remaining)), random_state=seed)
            corpus_pool = pd.concat([required, extra], ignore_index=True)
        else:
            corpus_pool = required.copy()

    corpus_pool = corpus_pool.drop_duplicates("doc_id").reset_index(drop=True)
    corpus_pool["kind"] = "corpus"
    corpus_pool["item_id"] = corpus_pool["doc_id"]
    corpus_pool["query_id"] = pd.NA
    corpus_pool["prefix"] = "passage: "
    corpus_pool["prefixed_text"] = corpus_pool["prefix"] + corpus_pool["text"].astype(str)

    pool_df = pd.concat(
        [
            query_pool[["kind", "item_id", "query_id", "doc_id", "text", "prefixed_text"]],
            corpus_pool[["kind", "item_id", "query_id", "doc_id", "text", "prefixed_text"]],
        ],
        ignore_index=True,
    )
    pool_df["row_id"] = np.arange(len(pool_df))
    return pool_df


def encode_pool(model: SentenceTransformer, pool_df: pd.DataFrame, batch_size: int = ENCODE_BATCH_SIZE) -> np.ndarray:
    embeddings = model.encode(
        pool_df["prefixed_text"].tolist(),
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
        device=DEVICE,
    )
    return np.asarray(embeddings, dtype=np.float32)


def split_pool_views(pool_df: pd.DataFrame, embeddings: np.ndarray) -> dict[str, Any]:
    kind = pool_df["kind"].astype(str).to_numpy()
    query_mask = kind == "query"
    corpus_mask = kind == "corpus"
    return {
        "pool_df": pool_df.reset_index(drop=True),
        "embeddings": np.asarray(embeddings, dtype=np.float32),
        "query_df": pool_df.loc[query_mask].reset_index(drop=True),
        "corpus_df": pool_df.loc[corpus_mask].reset_index(drop=True),
        "query_embeddings": np.asarray(embeddings, dtype=np.float32)[query_mask],
        "corpus_embeddings": np.asarray(embeddings, dtype=np.float32)[corpus_mask],
        "query_mask": query_mask,
        "corpus_mask": corpus_mask,
    }


def compute_local_id_frame(
    embeddings: np.ndarray,
    estimators: Sequence[str] = LOCAL_ID_ESTIMATORS,
    n_neighbors: int = LOCAL_ID_NEIGHBORS,
    n_jobs: int = -1,
) -> pd.DataFrame:
    emb = np.asarray(embeddings, dtype=np.float32)
    if emb.ndim != 2 or emb.shape[0] == 0:
        raise ValueError(f"Expected a 2-D embedding matrix, got shape {emb.shape}")
    n_neighbors = max(1, min(int(n_neighbors), max(1, emb.shape[0] - 1)))

    frame = pd.DataFrame(index=np.arange(emb.shape[0]))
    score_cols: List[str] = []
    for name in estimators:
        estimator = getattr(skdim.id, name)()
        dims = np.asarray(
            estimator.fit_transform_pw(emb, n_neighbors=n_neighbors, n_jobs=n_jobs),
            dtype=np.float32,
        ).reshape(-1)
        finite = np.isfinite(dims)
        if not finite.all():
            replacement = float(np.nanmedian(dims[finite])) if finite.any() else 0.0
            dims = np.where(finite, dims, replacement).astype(np.float32)
        col = f"local_id_{name.lower()}"
        frame[col] = dims
        score_cols.append(col)
    frame["local_id_mean"] = frame[score_cols].mean(axis=1)
    frame["local_id_median"] = frame[score_cols].median(axis=1)
    standardized = (frame[score_cols] - frame[score_cols].mean()) / (frame[score_cols].std(ddof=0) + 1e-8)
    frame["local_id_zmean"] = standardized.mean(axis=1)
    return frame


def local_score_thresholds(scores: pd.Series | np.ndarray, quantiles: Sequence[float] = THRESHOLD_QUANTILES) -> list[float]:
    arr = np.asarray(scores, dtype=np.float32)
    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        return []
    values = np.unique(np.quantile(finite, quantiles))
    return [float(v) for v in values]


def make_keep_mask(scores: pd.Series | np.ndarray, threshold: float, keep_high: bool = True) -> np.ndarray:
    arr = np.asarray(scores, dtype=np.float32)
    if keep_high:
        return arr >= threshold
    return arr <= threshold


def evaluate_retrieval_matrix(
    query_df: pd.DataFrame,
    corpus_df: pd.DataFrame,
    score_matrix: np.ndarray,
    qrels_map: Dict[str, Dict[str, int]],
    top_k: int = TOP_K,
    cutoffs: Sequence[int] = EVAL_CUTOFFS,
) -> tuple[Dict[str, float], pd.DataFrame, Dict[str, Dict[str, float]], Dict[str, Dict[str, int]]]:
    if len(query_df) == 0 or len(corpus_df) == 0:
        return {"ndcg@10": 0.0, "map@10": 0.0, "mrr": 0.0, "n_eval_queries": 0}, pd.DataFrame(), {}, {}

    query_ids = query_df["query_id"].astype(str).tolist()
    doc_ids = corpus_df["doc_id"].astype(str).tolist()
    corpus_id_set = set(doc_ids)

    qrels = {
        qid: {doc_id: int(score) for doc_id, score in qrels_map.get(qid, {}).items() if doc_id in corpus_id_set and int(score) > 0}
        for qid in query_ids
    }
    qrels = {qid: rels for qid, rels in qrels.items() if rels}
    if not qrels:
        return {"ndcg@10": 0.0, "map@10": 0.0, "mrr": 0.0, "n_eval_queries": 0}, pd.DataFrame(), {}, {}

    valid_q_mask = query_df["query_id"].isin(qrels.keys()).to_numpy()
    query_df = query_df.loc[valid_q_mask].reset_index(drop=True)
    score_matrix = np.asarray(score_matrix, dtype=np.float32)[valid_q_mask]
    query_ids = query_df["query_id"].astype(str).tolist()

    metric_names = {"recip_rank"}
    for cutoff in cutoffs:
        metric_names.update({f"ndcg_cut.{cutoff}", f"map_cut.{cutoff}", f"recall.{cutoff}", f"P.{cutoff}"})
    evaluator = pytrec_eval.RelevanceEvaluator(qrels, metric_names)

    top_k = min(int(top_k), score_matrix.shape[1])
    if top_k <= 0:
        return {"ndcg@10": 0.0, "map@10": 0.0, "mrr": 0.0, "n_eval_queries": 0}, pd.DataFrame(), {}, qrels

    if top_k < score_matrix.shape[1]:
        idx = np.argpartition(-score_matrix, kth=top_k - 1, axis=1)[:, :top_k]
    else:
        idx = np.tile(np.arange(score_matrix.shape[1]), (score_matrix.shape[0], 1))
    row_scores = np.take_along_axis(score_matrix, idx, axis=1)
    order = np.argsort(-row_scores, axis=1)
    idx = np.take_along_axis(idx, order, axis=1)
    row_scores = np.take_along_axis(row_scores, order, axis=1)

    run: Dict[str, Dict[str, float]] = {}
    for i, qid in enumerate(query_ids):
        run[qid] = {doc_ids[j]: float(s) for j, s in zip(idx[i], row_scores[i])}

    per_query = evaluator.evaluate(run)
    per_query_df = pd.DataFrame.from_dict(per_query, orient="index").fillna(0.0)
    summary = {pretty_pytrec_key(col): float(per_query_df[col].mean()) for col in per_query_df.columns}
    summary.update(
        {
            "n_eval_queries": int(len(qrels)),
            "n_query_rows": int(len(query_df)),
            "n_corpus_rows": int(len(corpus_df)),
            "top_k": int(top_k),
            "coverage_queries": float(len(qrels) / max(len(query_df), 1)),
        }
    )
    return summary, per_query_df, run, qrels


def pretty_pytrec_key(key: str) -> str:
    if key == "recip_rank":
        return "mrr"
    if key.startswith("ndcg_cut_"):
        return f"ndcg@{key.rsplit('_', 1)[-1]}"
    if key.startswith("map_cut_"):
        return f"map@{key.rsplit('_', 1)[-1]}"
    if key.startswith("recall_"):
        return f"recall@{key.rsplit('_', 1)[-1]}"
    if key.startswith("P_"):
        return f"precision@{key.rsplit('_', 1)[-1]}"
    return key


def evaluate_subset_from_masks(
    query_df: pd.DataFrame,
    corpus_df: pd.DataFrame,
    query_embeddings: np.ndarray,
    corpus_embeddings: np.ndarray,
    score_matrix: np.ndarray,
    query_keep: np.ndarray,
    corpus_keep: np.ndarray,
    qrels_map: Dict[str, Dict[str, int]],
    top_k: int = TOP_K,
    cutoffs: Sequence[int] = EVAL_CUTOFFS,
) -> tuple[Dict[str, float], dict[str, Any]]:
    query_keep = np.asarray(query_keep, dtype=bool)
    corpus_keep = np.asarray(corpus_keep, dtype=bool)
    sub_query_df = query_df.loc[query_keep].reset_index(drop=True)
    sub_corpus_df = corpus_df.loc[corpus_keep].reset_index(drop=True)
    sub_query_embeddings = np.asarray(query_embeddings, dtype=np.float32)[query_keep]
    sub_corpus_embeddings = np.asarray(corpus_embeddings, dtype=np.float32)[corpus_keep]
    sub_scores = np.asarray(score_matrix, dtype=np.float32)[np.ix_(query_keep, corpus_keep)]

    summary, per_query_df, run, qrels = evaluate_retrieval_matrix(
        sub_query_df,
        sub_corpus_df,
        sub_scores,
        qrels_map,
        top_k=top_k,
        cutoffs=cutoffs,
    )
    summary.update(
        {
            "kept_rows": int(len(sub_query_df) + len(sub_corpus_df)),
            "kept_queries": int(len(sub_query_df)),
            "kept_corpus": int(len(sub_corpus_df)),
            "kept_fraction": float((len(sub_query_df) + len(sub_corpus_df)) / max(len(query_df) + len(corpus_df), 1)),
            "kept_query_fraction": float(len(sub_query_df) / max(len(query_df), 1)),
            "kept_corpus_fraction": float(len(sub_corpus_df) / max(len(corpus_df), 1)),
        }
    )
    bundle = {
        "query_df": sub_query_df,
        "corpus_df": sub_corpus_df,
        "query_embeddings": sub_query_embeddings,
        "corpus_embeddings": sub_corpus_embeddings,
        "score_matrix": sub_scores,
        "per_query_df": per_query_df,
        "run": run,
        "qrels": qrels,
    }
    return summary, bundle


def compute_unsup_diagnostics(
    query_embeddings: np.ndarray,
    corpus_embeddings: np.ndarray,
    pool_embeddings: np.ndarray,
    diag_sample_size: int = DIAG_INTRINSIC_SAMPLE_SIZE,
) -> Dict[str, float]:
    diagnostics: Dict[str, float] = {}
    if len(query_embeddings) > 1 and len(corpus_embeddings) > 1:
        retr_metrics = compute_metrics_retrieval(
            query_embs=np.asarray(query_embeddings, dtype=np.float32),
            corpus_embs=np.asarray(corpus_embeddings, dtype=np.float32),
            selected_metrics=[
                "rankme",
                "coherence",
                "stable_rank",
                "intrinsic_dim_global_fast",
                "intrinsic_dim_local_fast",
            ],
            n_samples=1,
            sample_fraction=UNSUP_SAMPLE_FRACTION,
            min_sample_size=UNSUP_MIN_SAMPLE_SIZE,
            include_ph_dim=False,
            ripser_maxdim=0,
        )
        for variant, metric_dict in retr_metrics.items():
            diagnostics.update(suffix_metrics(metric_dict, variant))

    sample_size = min(int(diag_sample_size), len(pool_embeddings))
    diagnostics["diag_sample_size"] = float(sample_size)
    if sample_size < 2:
        return diagnostics

    rng = np.random.default_rng(SEED)
    sample_idx = rng.choice(len(pool_embeddings), size=sample_size, replace=False)
    sample = np.asarray(pool_embeddings[sample_idx], dtype=np.float32)
    try:
        raw_global = compute_intrinsic_dim_global(sample, estimator_names=list(FAST_GLOBAL_ESTIMATORS), verbose=False)
        diagnostics.update({f"raw_intrinsic_dim_global_{key}": float(value) for key, value in raw_global.items()})
    except Exception as exc:
        diagnostics["raw_intrinsic_dim_global_error"] = str(exc)
    try:
        raw_local = compute_intrinsic_dim_local(
            sample,
            estimator_names=list(FAST_LOCAL_ESTIMATORS),
            n_neighbors=min(LOCAL_ID_NEIGHBORS, max(1, sample.shape[0] - 1)),
            n_jobs=-1,
            verbose=False,
        )
        diagnostics.update({f"raw_intrinsic_dim_local_{key}": float(value) for key, value in raw_local.items()})
    except Exception as exc:
        diagnostics["raw_intrinsic_dim_local_error"] = str(exc)
    return diagnostics

In [ ]:
def save_dataframe(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def save_json(data: Dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True))


def plot_local_score_distribution(
    pool_df: pd.DataFrame,
    score_col: str,
    title: str,
    out_path: Optional[Path] = None,
):
    fig, ax = plt.subplots(figsize=(11, 4))
    sns.histplot(
        data=pool_df,
        x=score_col,
        hue="kind",
        bins=40,
        stat="density",
        common_norm=False,
        element="step",
        fill=False,
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel(score_col)
    ax.set_ylabel("density")
    fig.tight_layout()
    if out_path is not None:
        out_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out_path, dpi=160, bbox_inches="tight")
    return fig, ax


def plot_threshold_sweep(
    sweep_df: pd.DataFrame,
    title: str,
    out_path: Optional[Path] = None,
    metric_cols: Sequence[str] = ("ndcg@10", "map@10", "mrr"),
):
    fig, ax = plt.subplots(figsize=(11, 5))
    ordered = sweep_df.sort_values("kept_fraction")
    for col in metric_cols:
        if col in ordered.columns:
            ax.plot(ordered["kept_fraction"], ordered[col], marker="o", label=col)
    ax.set_xlabel("retained fraction")
    ax.set_ylabel("retrieval quality")
    ax.set_title(title)
    ax.legend(loc="best")
    fig.tight_layout()
    if out_path is not None:
        out_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out_path, dpi=160, bbox_inches="tight")
    return fig, ax


def noisy_view(text: str, rng: random.Random, drop_prob: float = 0.18) -> str:
    words = str(text).split()
    if len(words) <= 1:
        return str(text).strip()
    kept = [word for word in words if rng.random() > drop_prob]
    if not kept:
        kept = [rng.choice(words)]
    if len(kept) > 6 and rng.random() < 0.25:
        rng.shuffle(kept)
    return " ".join(kept).strip()


def build_self_supervised_pairs(
    texts: Sequence[str],
    max_examples: int = 2000,
    seed: int = SEED,
) -> list[InputExample]:
    rng = random.Random(seed)
    clean_texts = [str(text).strip() for text in texts if str(text).strip()]
    examples: list[InputExample] = []
    for text in clean_texts[:max_examples]:
        anchor = noisy_view(text, rng, drop_prob=0.05)
        positive = noisy_view(text, rng, drop_prob=0.25)
        if anchor and positive:
            examples.append(InputExample(texts=[anchor, positive]))
    return examples


def fine_tune_unsupervised_model(
    base_model_name: str,
    train_texts: Sequence[str],
    output_dir: Path,
    epochs: int = 1,
    batch_size: int = 16,
    max_examples: int = 2000,
) -> Optional[SentenceTransformer]:
    train_examples = build_self_supervised_pairs(train_texts, max_examples=max_examples, seed=SEED)
    if len(train_examples) < 8:
        print(f"Skipping adaptation: only {len(train_examples)} training pairs were built.")
        return None

    model = SentenceTransformer(base_model_name, device=DEVICE)
    train_loader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)
    train_loss = MultipleNegativesRankingLoss(model)
    warmup_steps = max(10, len(train_loader) // 10)
    output_dir.mkdir(parents=True, exist_ok=True)
    model.fit(
        train_objectives=[(train_loader, train_loss)],
        epochs=epochs,
        warmup_steps=warmup_steps,
        output_path=str(output_dir),
        save_best_model=False,
        show_progress_bar=True,
    )
    return model

In [ ]:
from IPython.display import display

SCORE_COL = "local_id_mean"
BEST_METRIC = "ndcg@10"
KEEP_HIGH = KEEP_HIGH_SCORES

model = SentenceTransformer(MODEL_NAME, device=DEVICE)

task_artifacts: Dict[str, Dict[str, Any]] = {}
baseline_rows: list[Dict[str, Any]] = []
filtered_rows: list[Dict[str, Any]] = []
sweep_rows: list[Dict[str, Any]] = []

for config in TASK_CONFIGS:
    print(f"\n=== {config.task_name} ===")
    task_dir = RUN_DIR / config.task_name
    plot_dir = task_dir / "plots"
    task_dir.mkdir(parents=True, exist_ok=True)
    plot_dir.mkdir(parents=True, exist_ok=True)

    queries_df, corpus_df, qrels_df = load_retrieval_tables(config)
    qrels_map = build_qrels_map(qrels_df)
    pool_df = build_analysis_pool(config.task_name, queries_df, corpus_df, qrels_map, corpus_cap=config.corpus_cap, seed=SEED)

    embeddings = encode_pool(model, pool_df, batch_size=ENCODE_BATCH_SIZE)
    local_df = compute_local_id_frame(
        embeddings,
        estimators=LOCAL_ID_ESTIMATORS,
        n_neighbors=LOCAL_ID_NEIGHBORS,
        n_jobs=-1,
    )
    pool_df = pd.concat([pool_df.reset_index(drop=True), local_df.reset_index(drop=True)], axis=1)
    pool_df[SCORE_COL] = pool_df["local_id_mean"]
    save_dataframe(pool_df, task_dir / "pool_with_local_scores.csv")

    query_count = int((pool_df["kind"] == "query").sum())
    query_df = pool_df.iloc[:query_count].reset_index(drop=True)
    corpus_df = pool_df.iloc[query_count:].reset_index(drop=True)
    query_embeddings = embeddings[:query_count]
    corpus_embeddings = embeddings[query_count:]
    score_matrix = query_embeddings @ corpus_embeddings.T

    plot_local_score_distribution(
        pool_df,
        SCORE_COL,
        title=f"{config.task_name}: raw local-ID scores",
        out_path=plot_dir / "local_score_distribution.png",
    )

    baseline_summary, baseline_bundle = evaluate_retrieval_matrix(
        query_df=query_df,
        corpus_df=corpus_df,
        score_matrix=score_matrix,
        qrels_map=qrels_map,
        top_k=TOP_K,
        cutoffs=EVAL_CUTOFFS,
    )
    baseline_diag = compute_unsup_diagnostics(query_embeddings, corpus_embeddings, embeddings)
    baseline_row: Dict[str, Any] = {
        "task_name": config.task_name,
        "stage": "baseline",
        "threshold": float("nan"),
        "threshold_policy": "none",
        "score_col": SCORE_COL,
        "keep_high": KEEP_HIGH,
        "kept_rows": int(len(query_df) + len(corpus_df)),
        "kept_queries": int(len(query_df)),
        "kept_corpus": int(len(corpus_df)),
        "kept_fraction": 1.0,
        "kept_query_fraction": 1.0,
        "kept_corpus_fraction": 1.0,
        **baseline_summary,
    }
    baseline_row.update({f"diag_{key}": value for key, value in baseline_diag.items()})
    baseline_rows.append(baseline_row)
    save_dataframe(pd.DataFrame([baseline_row]), task_dir / "baseline_summary.csv")

    thresholds = local_score_thresholds(pool_df[SCORE_COL], THRESHOLD_QUANTILES)
    task_sweep_rows: list[Dict[str, Any]] = []
    for threshold in thresholds:
        keep_mask = make_keep_mask(pool_df[SCORE_COL], threshold, keep_high=KEEP_HIGH)
        summary, bundle = evaluate_subset_from_masks(
            query_df=query_df,
            corpus_df=corpus_df,
            query_embeddings=query_embeddings,
            corpus_embeddings=corpus_embeddings,
            score_matrix=score_matrix,
            query_keep=keep_mask[:query_count],
            corpus_keep=keep_mask[query_count:],
            qrels_map=qrels_map,
            top_k=TOP_K,
            cutoffs=EVAL_CUTOFFS,
        )
        summary.update(
            {
                "task_name": config.task_name,
                "stage": "threshold_sweep",
                "threshold": float(threshold),
                "threshold_policy": "keep_high" if KEEP_HIGH else "keep_low",
                "score_col": SCORE_COL,
                "keep_high": KEEP_HIGH,
            }
        )
        task_sweep_rows.append(summary)
        sweep_rows.append(summary)

    sweep_df = pd.DataFrame(task_sweep_rows).sort_values("kept_fraction", ascending=False).reset_index(drop=True)
    save_dataframe(sweep_df, task_dir / "threshold_sweep.csv")
    plot_threshold_sweep(
        sweep_df,
        title=f"{config.task_name}: retained fraction vs retrieval quality",
        out_path=plot_dir / "threshold_sweep.png",
    )

    best_idx = sweep_df[BEST_METRIC].astype(float).idxmax()
    best_threshold = float(sweep_df.loc[best_idx, "threshold"])
    best_keep_mask = make_keep_mask(pool_df[SCORE_COL], best_threshold, keep_high=KEEP_HIGH)
    best_summary, best_bundle = evaluate_subset_from_masks(
        query_df=query_df,
        corpus_df=corpus_df,
        query_embeddings=query_embeddings,
        corpus_embeddings=corpus_embeddings,
        score_matrix=score_matrix,
        query_keep=best_keep_mask[:query_count],
        corpus_keep=best_keep_mask[query_count:],
        qrels_map=qrels_map,
        top_k=TOP_K,
        cutoffs=EVAL_CUTOFFS,
    )
    best_diag = compute_unsup_diagnostics(
        best_bundle["query_embeddings"],
        best_bundle["corpus_embeddings"],
        np.concatenate([best_bundle["query_embeddings"], best_bundle["corpus_embeddings"]], axis=0),
    )
    best_row: Dict[str, Any] = {
        "task_name": config.task_name,
        "stage": "best_filtered",
        "threshold": best_threshold,
        "threshold_policy": "keep_high" if KEEP_HIGH else "keep_low",
        "score_col": SCORE_COL,
        "keep_high": KEEP_HIGH,
        **best_summary,
    }
    best_row.update({f"diag_{key}": value for key, value in best_diag.items()})
    filtered_rows.append(best_row)
    save_dataframe(pd.DataFrame([best_row]), task_dir / "best_filtered_summary.csv")
    save_dataframe(pd.DataFrame([best_diag]), task_dir / "best_filtered_diagnostics.csv")

    task_artifacts[config.task_name] = {
        "config": config,
        "task_dir": task_dir,
        "plot_dir": plot_dir,
        "queries_df": query_df,
        "corpus_df": corpus_df,
        "pool_df": pool_df,
        "embeddings": embeddings,
        "query_embeddings": query_embeddings,
        "corpus_embeddings": corpus_embeddings,
        "score_matrix": score_matrix,
        "qrels_df": qrels_df,
        "qrels_map": qrels_map,
        "baseline_row": baseline_row,
        "baseline_diag": baseline_diag,
        "baseline_bundle": baseline_bundle,
        "sweep_df": sweep_df,
        "best_row": best_row,
        "best_diag": best_diag,
        "best_bundle": best_bundle,
        "best_threshold": best_threshold,
        "best_keep_mask": best_keep_mask,
    }

baseline_df = pd.DataFrame(baseline_rows)
filtered_df = pd.DataFrame(filtered_rows)
sweep_all_df = pd.DataFrame(sweep_rows)
comparison_df = pd.concat([baseline_df, filtered_df], ignore_index=True)

save_dataframe(baseline_df, RUN_DIR / "baseline_results.csv")
save_dataframe(filtered_df, RUN_DIR / "best_filtered_results.csv")
save_dataframe(sweep_all_df, RUN_DIR / "threshold_sweeps_all.csv")
save_dataframe(comparison_df, RUN_DIR / "comparison_results.csv")

summary_view = comparison_df[
    [
        "task_name",
        "stage",
        "threshold",
        "kept_fraction",
        "kept_query_fraction",
        "kept_corpus_fraction",
        "ndcg@10",
        "map@10",
        "mrr",
        "coverage_queries",
        "n_eval_queries",
    ]
].copy()

print("\nSummary view")
display(summary_view)

run_manifest = {
    "run_name": RUN_NAME,
    "model_name": MODEL_NAME,
    "device": DEVICE,
    "score_col": SCORE_COL,
    "keep_high": KEEP_HIGH,
    "best_metric": BEST_METRIC,
    "threshold_quantiles": [float(q) for q in THRESHOLD_QUANTILES.tolist()],
    "tasks": [config.task_name for config in TASK_CONFIGS],
    "corpus_caps": {config.task_name: config.corpus_cap for config in TASK_CONFIGS},
}
save_json(run_manifest, RUN_DIR / "run_manifest.json")

In [ ]:
RUN_ADAPTATION = False
ADAPTATION_TASK = "FiQA2018"
ADAPTATION_TARGET_TASKS = ["AskUbuntuDupQuestions", "FiQA2018"]
ADAPTATION_QUANTILE = 0.25
ADAPTATION_EPOCHS = 1
ADAPTATION_BATCH_SIZE = 16
ADAPTATION_MAX_EXAMPLES = 1500

adaptation_rows: list[Dict[str, Any]] = []
adapted_task_artifacts: Dict[str, Dict[str, Any]] = {}

if RUN_ADAPTATION:
    anchor_artifact = task_artifacts[ADAPTATION_TASK]
    anchor_pool = anchor_artifact["pool_df"]
    anchor_best_threshold = float(anchor_artifact["best_threshold"])
    anchor_threshold = float(np.quantile(anchor_pool[SCORE_COL].to_numpy(dtype=np.float32), ADAPTATION_QUANTILE))
    low_mask = ~make_keep_mask(anchor_pool[SCORE_COL], anchor_threshold, keep_high=KEEP_HIGH)
    train_texts = anchor_pool.loc[low_mask, "text"].drop_duplicates().tolist()
    adapt_dir = RUN_DIR / "adapted_model"
    adapted_model = fine_tune_unsupervised_model(
        MODEL_NAME,
        train_texts,
        adapt_dir,
        epochs=ADAPTATION_EPOCHS,
        batch_size=ADAPTATION_BATCH_SIZE,
        max_examples=ADAPTATION_MAX_EXAMPLES,
    )

    if adapted_model is None:
        save_json(
            {
                "enabled": False,
                "reason": "insufficient self-supervised training pairs",
                "anchor_task": ADAPTATION_TASK,
                "anchor_threshold": anchor_threshold,
            },
            RUN_DIR / "adaptation_manifest.json",
        )
    else:
        def run_task_with_model(
            model: SentenceTransformer,
            config: RetrievalTaskConfig,
            anchor_threshold_value: float,
        ) -> list[Dict[str, Any]]:
            task_dir = RUN_DIR / "adaptation" / config.task_name
            plot_dir = task_dir / "plots"
            task_dir.mkdir(parents=True, exist_ok=True)
            plot_dir.mkdir(parents=True, exist_ok=True)

            queries_df, corpus_df, qrels_df = load_retrieval_tables(config)
            qrels_map = build_qrels_map(qrels_df)
            pool_df = build_analysis_pool(config.task_name, queries_df, corpus_df, qrels_map, corpus_cap=config.corpus_cap, seed=SEED)
            embeddings = encode_pool(model, pool_df, batch_size=ENCODE_BATCH_SIZE)
            local_df = compute_local_id_frame(
                embeddings,
                estimators=LOCAL_ID_ESTIMATORS,
                n_neighbors=LOCAL_ID_NEIGHBORS,
                n_jobs=-1,
            )
            pool_df = pd.concat([pool_df.reset_index(drop=True), local_df.reset_index(drop=True)], axis=1)
            pool_df[SCORE_COL] = pool_df["local_id_mean"]
            save_dataframe(pool_df, task_dir / "pool_with_local_scores.csv")
            plot_local_score_distribution(
                pool_df,
                SCORE_COL,
                title=f"{config.task_name}: adapted raw local-ID scores",
                out_path=plot_dir / "local_score_distribution.png",
            )

            query_count = int((pool_df["kind"] == "query").sum())
            query_df = pool_df.iloc[:query_count].reset_index(drop=True)
            corpus_df = pool_df.iloc[query_count:].reset_index(drop=True)
            query_embeddings = embeddings[:query_count]
            corpus_embeddings = embeddings[query_count:]
            score_matrix = query_embeddings @ corpus_embeddings.T

            baseline_summary, baseline_bundle = evaluate_retrieval_matrix(
                query_df=query_df,
                corpus_df=corpus_df,
                score_matrix=score_matrix,
                qrels_map=qrels_map,
                top_k=TOP_K,
                cutoffs=EVAL_CUTOFFS,
            )
            baseline_diag = compute_unsup_diagnostics(query_embeddings, corpus_embeddings, embeddings)
            baseline_row: Dict[str, Any] = {
                "task_name": config.task_name,
                "stage": "adapted_baseline",
                "threshold": float("nan"),
                "threshold_policy": "none",
                "score_col": SCORE_COL,
                "keep_high": KEEP_HIGH,
                "kept_rows": int(len(query_df) + len(corpus_df)),
                "kept_queries": int(len(query_df)),
                "kept_corpus": int(len(corpus_df)),
                "kept_fraction": 1.0,
                "kept_query_fraction": 1.0,
                "kept_corpus_fraction": 1.0,
                **baseline_summary,
            }
            baseline_row.update({f"diag_{key}": value for key, value in baseline_diag.items()})

            filtered_row: Dict[str, Any] | None = None
            if config.task_name in ADAPTATION_TARGET_TASKS:
                keep_mask = make_keep_mask(pool_df[SCORE_COL], anchor_threshold_value, keep_high=KEEP_HIGH)
                filtered_summary, filtered_bundle = evaluate_subset_from_masks(
                    query_df=query_df,
                    corpus_df=corpus_df,
                    query_embeddings=query_embeddings,
                    corpus_embeddings=corpus_embeddings,
                    score_matrix=score_matrix,
                    query_keep=keep_mask[:query_count],
                    corpus_keep=keep_mask[query_count:],
                    qrels_map=qrels_map,
                    top_k=TOP_K,
                    cutoffs=EVAL_CUTOFFS,
                )
                filtered_diag = compute_unsup_diagnostics(
                    filtered_bundle["query_embeddings"],
                    filtered_bundle["corpus_embeddings"],
                    np.concatenate([filtered_bundle["query_embeddings"], filtered_bundle["corpus_embeddings"]], axis=0),
                )
                filtered_row = {
                    "task_name": config.task_name,
                    "stage": "adapted_filtered",
                    "threshold": anchor_threshold_value,
                    "threshold_policy": "keep_high" if KEEP_HIGH else "keep_low",
                    "score_col": SCORE_COL,
                    "keep_high": KEEP_HIGH,
                    **filtered_summary,
                }
                filtered_row.update({f"diag_{key}": value for key, value in filtered_diag.items()})
                adaptation_rows.append(filtered_row)
                adapted_task_artifacts[config.task_name] = {
                    "config": config,
                    "task_dir": task_dir,
                    "plot_dir": plot_dir,
                    "pool_df": pool_df,
                    "embeddings": embeddings,
                    "query_embeddings": query_embeddings,
                    "corpus_embeddings": corpus_embeddings,
                    "score_matrix": score_matrix,
                    "baseline_row": baseline_row,
                    "baseline_diag": baseline_diag,
                    "baseline_bundle": baseline_bundle,
                    "filtered_row": filtered_row,
                    "filtered_diag": filtered_diag,
                    "filtered_bundle": filtered_bundle,
                    "anchor_threshold": anchor_threshold_value,
                }
            else:
                adapted_task_artifacts[config.task_name] = {
                    "config": config,
                    "task_dir": task_dir,
                    "plot_dir": plot_dir,
                    "pool_df": pool_df,
                    "embeddings": embeddings,
                    "query_embeddings": query_embeddings,
                    "corpus_embeddings": corpus_embeddings,
                    "score_matrix": score_matrix,
                    "baseline_row": baseline_row,
                    "baseline_diag": baseline_diag,
                    "baseline_bundle": baseline_bundle,
                    "anchor_threshold": anchor_threshold_value,
                }

            save_dataframe(pd.DataFrame([baseline_row]), task_dir / "adapted_baseline_summary.csv")
            if filtered_row is not None:
                save_dataframe(pd.DataFrame([filtered_row]), task_dir / "adapted_filtered_summary.csv")
            return [baseline_row] + ([filtered_row] if filtered_row is not None else [])

        for config in TASK_CONFIGS:
            adaptation_rows.extend(run_task_with_model(adapted_model, config, anchor_threshold))

        adaptation_df = pd.DataFrame(adaptation_rows)
        save_dataframe(adaptation_df, RUN_DIR / "adaptation_results.csv")
        save_json(
            {
                "enabled": True,
                "anchor_task": ADAPTATION_TASK,
                "anchor_threshold": anchor_threshold,
                "anchor_best_threshold": anchor_best_threshold,
                "target_tasks": ADAPTATION_TARGET_TASKS,
                "epochs": ADAPTATION_EPOCHS,
                "batch_size": ADAPTATION_BATCH_SIZE,
                "max_examples": ADAPTATION_MAX_EXAMPLES,
            },
            RUN_DIR / "adaptation_manifest.json",
        )
        display(
            adaptation_df[
                [
                    "task_name",
                    "stage",
                    "threshold",
                    "kept_fraction",
                    "kept_query_fraction",
                    "kept_corpus_fraction",
                    "ndcg@10",
                    "map@10",
                    "mrr",
                    "coverage_queries",
                    "n_eval_queries",
                ]
            ]
        )
else:
    save_json(
        {
            "enabled": False,
            "anchor_task": ADAPTATION_TASK,
            "anchor_best_threshold": float(task_artifacts[ADAPTATION_TASK]["best_threshold"]),
            "anchor_threshold": float(np.quantile(task_artifacts[ADAPTATION_TASK]["pool_df"][SCORE_COL].to_numpy(dtype=np.float32), ADAPTATION_QUANTILE)),
            "target_tasks": ADAPTATION_TARGET_TASKS,
            "epochs": ADAPTATION_EPOCHS,
            "batch_size": ADAPTATION_BATCH_SIZE,
            "max_examples": ADAPTATION_MAX_EXAMPLES,
            "note": "set RUN_ADAPTATION = True to fine-tune on low-metric OOD texts",
        },
        RUN_DIR / "adaptation_manifest.json",
    )
    print("Adaptation disabled; set RUN_ADAPTATION = True to rerun the optional self-supervised step.")

adaptation_df = pd.DataFrame(adaptation_rows)
if not adaptation_df.empty:
    save_dataframe(adaptation_df, RUN_DIR / "adaptation_results.csv")

final_rows = baseline_rows + filtered_rows + adaptation_rows
final_summary_df = pd.DataFrame(final_rows)
save_dataframe(final_summary_df, RUN_DIR / "final_summary.csv")
print(f"Saved final summary to {RUN_DIR / 'final_summary.csv'}")